# 02 · Inject label noise (training labels only)

**What this step does.** Makes corrupted copies of the training labels at seven doses, three mechanisms and five seeds. We record **exactly which records were flipped**, so detection can be scored like any detector.

| Mechanism | What is flipped | Real-world analogue |
|---|---|---|
| uniform | a random share of ALL labels, either direction | careless data entry |
| class_conditional | a share of POSITIVE labels turned negative, everyone | readmissions at another hospital that never reach the record; defaults never reported |
| targeted | a share of POSITIVE labels turned negative, ONLY in the risk stratum | the same, concentrated in one group |

**Dose** = share of the eligible pool that is flipped (0.00 … 0.40). Because pools differ in size, we also record the **global** share of labels changed: a 0.40 targeted dose may change only ~1% of all labels. That gap is the heart of H2.

**Inputs:** `data/{dataset}_train.csv`, `stage0_out/risk_strata_frozen.json` · **Outputs:** `results/noisy/{dataset}__{mechanism}__{dose}__s{seed}.csv` (index, clean label, noisy label, flipped, in-stratum), `results/02_inject_log.csv`.

**Which datasets run:** `li.RUN_ONLY` in `li_lib.py` (now: Taiwan credit only). Results for the other datasets are kept: the CSVs are merged, not overwritten.

In [1]:
# --- setup: run this cell first -------------------------------------------
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))          # li_lib.py sits next to the notebooks
import li_lib as li
pd.set_option("display.width", 160, "display.max_columns", 30)
print("li_lib", li.VERSION, "| project folder:", li.ROOT)
li.versions()

li_lib 1.1.0 | project folder: D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity


,package,version
0,numpy,2.2.6
1,pandas,3.0.6
2,scikit-learn,1.9.1
3,xgboost,3.4.1
4,cleanlab,2.9.0
5,lingam,1.13.0
6,causal-learn,0.1.4.8


In [2]:
(li.RESULTS / "noisy").mkdir(exist_ok=True)
log = []
for name, cfg in li.run_configs().items():
    cfg = li.apply_frozen(cfg)
    tr = li.load_split(cfg, "train")
    y = tr[cfg.label].to_numpy(int)
    s = li.stratum_mask(tr, cfg)
    for mech in li.MECHANISMS:
        for dose in li.DOSES:
            for seed in li.SEEDS:
                noisy, flipped, info = li.inject(y, mech, dose, seed, s)
                pd.DataFrame({"row": np.arange(len(y)), "y_clean": y, "y_noisy": noisy,
                              "flipped": flipped, "in_stratum": s}).to_csv(
                    li.RESULTS / "noisy" / f"{name}__{mech}__{dose:.2f}__s{seed}.csv", index=False)
                log.append({"dataset": name, "stratum": cfg.stratum_name, **info})
inject_log = pd.DataFrame(log)
print(len(inject_log), "corrupted training sets written")
inject_log = li.save_merged(inject_log, "02_inject_log.csv", show=0)

105 corrupted training sets written
kept 315 existing rows for ['cdcdiabetes', 'diabetes130', 'southgerman']
wrote D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\label_integrity\results\02_inject_log.csv  (420 rows x 12 cols)


## Check: did we inject what we meant to?
**Look for:** `realized_pool_fraction` equals the dose (within rounding). Compare `realized_global_fraction` across mechanisms: targeted noise changes far fewer labels overall.

In [3]:
chk = (inject_log.groupby(["dataset", "mechanism", "dose"])
       [["pool_size", "realized_pool_fraction", "realized_global_fraction", "realized_stratum_fraction"]]
       .mean().round(4))
chk

pool_size  realized_pool_fraction  realized_global_fraction  realized_stratum_fraction
dataset      mechanism         dose                                                                                        
cdcdiabetes  class_conditional 0.00     4697.0                  0.0000                    0.0000                     0.0000
                               0.03     4697.0                  0.0300                    0.0047                     0.0054
                               0.05     4697.0                  0.0500                    0.0078                     0.0073
                               0.10     4697.0                  0.1001                    0.0157                     0.0156
                               0.20     4697.0                  0.1999                    0.0313                     0.0316
...                                        ...                     ...                       ...                        ...
taiwancredit uniform           0.05    17979.0                  0.0500                    0.0500                     0.0483
                               0.10    17979.0                  0.1000                    0.1000                     0.0974
                               0.20    17979.0                  0.2000                    0.2000                     0.1980
                               0.30    17979.0                  0.3000                    0.3000                     0.2988
                               0.40    17979.0                  0.4000                    0.4000                     0.3966

[84 rows x 4 columns]

## What to look for before moving on
- [ ] Realized pool fraction matches each dose.
- [ ] At dose 0.40, targeted noise changes only a small global share of labels, but a large share inside the stratum.

**For the paper:** the mechanism table above goes in Experimental Design.